In [4]:
import logging



In [5]:

import sys

sys.path.append('E:\wangzhilin\QuantumGalaxy')
print(sys.path)

['e:\\wangzhilin\\QuantumGalaxy\\opt', 'e:\\ProgramData\\Anaconda3\\envs\\QGIBG\\python38.zip', 'e:\\ProgramData\\Anaconda3\\envs\\QGIBG\\DLLs', 'e:\\ProgramData\\Anaconda3\\envs\\QGIBG\\lib', 'e:\\ProgramData\\Anaconda3\\envs\\QGIBG', '', 'e:\\ProgramData\\Anaconda3\\envs\\QGIBG\\lib\\site-packages', 'E:\\ProgramData\\THSDataInterface_Windows\\bin\\x64', 'e:\\ProgramData\\Anaconda3\\envs\\QGIBG\\lib\\site-packages\\win32', 'e:\\ProgramData\\Anaconda3\\envs\\QGIBG\\lib\\site-packages\\win32\\lib', 'e:\\ProgramData\\Anaconda3\\envs\\QGIBG\\lib\\site-packages\\Pythonwin', 'E:\\wangzhilin\\QuantumGalaxy']


In [6]:

from QGI.mysql import MYSQL


In [7]:
from QGI.feishu import *

In [8]:
from iFinDPy import *

e:\ProgramData\Anaconda3\envs\QGIBG\Lib\site-packages\iFinDPy.pth


In [9]:

def get_logger():
    logger=logging.getLogger('logger')
    logger.setLevel(logging.INFO)
    fh=logging.FileHandler("E:/wangzhilin/QuantumGalaxy/logs/n_log.log",'a')
    fh.setLevel(logging.INFO)
    ch=logging.StreamHandler()
    ch.setLevel(logging.DEBUG)
    formatter=logging.Formatter(fmt='%(asctime)s %(name)-12s %(levelname)-8s %(message)s',datefmt='%m-%d %H:%M')
    ch.setFormatter(formatter)
    fh.setFormatter(formatter)
    logger.addHandler(ch)
    logger.addHandler(fh)
    return logger


In [17]:
host='localhost'
user='Local_Editor'
password='QuantumGalaxy'
database='qgdbs'
mysql=MYSQL(host,user,password,database)


In [22]:
def get_query_list(mysql):

    sql0="select * from edb_info where name like '%国债%' and name like '%收益率%'"
    result_tuple=mysql.read_query(sql0)
    result=[]
    for r in result_tuple:
        result.append((r[1],r[3]))#(code,name)

    
    return(result)


In [23]:
r=get_query_list(mysql)

In [31]:
r

[('L001619976', '国债到期收益率:2年'),
 ('L001618529', '国债到期收益率:3年'),
 ('L001619472', '国债到期收益率:5年'),
 ('G002600770', '美国:国债收益率:2年'),
 ('G002600771', '美国:国债收益率:3年'),
 ('G002600772', '美国:国债收益率:5年'),
 ('L001618528', '国债到期收益率:1年'),
 ('G002600769', '美国:国债收益率:1年'),
 ('L001619518', '国债到期收益率:10年'),
 ('G002600774', '美国:国债收益率:10年')]

In [19]:
day=date.today()-timedelta(1)
day=day.strftime('%Y-%m-%d')
day

'2022-05-23'

In [30]:
def query(result,day):
    THS_iFinDLogin('lzxh0011','450503')

    edb_list=[]
    for row in result:
            edb_list.append(row[0])
    edb_str=''
    for s in edb_list:
        edb_str+=s+';'
    edb_str=edb_str[:-1]
    
    data=THS_EDB(edb_str,'',day,day)
    THS_iFinDLogout()
    return data


In [47]:
data=query(r,'2022-05-23')
data

errorcode=0
errmsg=
data=         time          id   value    index_name                rtime
0  2022-05-23  L001619976  2.2494    国债到期收益率:2年  2022-05-23 00:00:00
1  2022-05-23  L001618529  2.3299    国债到期收益率:3年  2022-05-23 00:00:00
2  2022-05-23  L001619472  2.5252    国债到期收益率:5年  2022-05-23 00:00:00
3  2022-05-23  G002600770    2.65   美国:国债收益率:2年  2022-05-24 05:15:43
4  2022-05-23  G002600771     2.8   美国:国债收益率:3年  2022-05-24 05:15:43
5  2022-05-23  G002600772    2.88   美国:国债收益率:5年  2022-05-24 05:15:43
6  2022-05-23  L001618528  1.9489    国债到期收益率:1年  2022-05-23 00:00:00
7  2022-05-23  G002600769    2.09   美国:国债收益率:1年  2022-05-24 05:15:43
8  2022-05-23  L001619518  2.7709   国债到期收益率:10年  2022-05-23 00:00:00
9  2022-05-23  G002600774    2.86  美国:国债收益率:10年  2022-05-24 05:15:43

In [48]:
val0=[]
for i,row in data.data.iterrows():
    
    val0.append((row['id'],row['time'],float(row['value'])))
sql0="insert ignore into edb_data (code,date,value) values(%s,%s,%s)"

result0=mysql.wirte_many_query(sql0,val0)
result0


2

In [15]:
def compute(data):
    debt2,debt3,debt5,usdebt2,usdebt3,usdebt5=data['value'].astype(float)/100
    debt25=(((1+debt5)**5)/((1+debt2)**2))**(1/3)-1.005
    usdebt25=(((1+usdebt5)**5)/((1+usdebt2)**2))**(1/3)-1.005
    n_now=(debt3)**-1
    n_25=(debt25)**-1
    n_usnow=usdebt3**-1
    n_us25=usdebt25**-1
    #顺序：国债-3 N-now 国债-2 国债-3  国债-2~5 N-2 美国债-3 美N-now 美国债-2 美国债-5 美国债-2~5 美N-2
    data={'n0_cny':n_now,'n2_cny':n_25,'n0_usd':n_usnow,'n2_usd':n_us25}
    return data

In [14]:
def feishu_work(mysql,day):
    sql='select name,value from customized_edb_data where find_in_set(name,"n0_cny,n2_cny,n0_usd,n2_usd") and date="'+day+'"'
    result=mysql.read_query(sql)
    print(sql)


    result=dict(result)
    datalist=[result['n0_cny'],result['n2_cny'],result['n0_usd'],result['n2_usd']]
    datalist=[day]+datalist
    r=append_table(datalist,table_id=table_dict['n_table_id'],sheet_id=table_dict['n_sheet1_id'])
    title='N值发布 - %s\n'%(date.today())
    msg='A股（参考中国国债收益率，以当下及两年后的预期收益率综合计算后得出）\n当下的N：%.3f\n两年后远期的N：%.3f\n'%(datalist[1],datalist[2])
    msg+='美股（参考美国国债收益率，以当下及两年后的预期收益率综合计算后得出）\n当下的N：%.3f\n两年后远期的N：%.3f\n'%(datalist[3],datalist[4])
    #r=post_msg(title,msg,webhook='https://open.feishu.cn/open-apis/bot/v2/hook/f04a0171-6a64-4215-bba9-e9e7ff5bce7d')
    #r=post_msg(title,msg,webhook=webhook_dict['全员群voyager'])
    return r


In [15]:
logger=get_logger()

In [2]:
def job():

    host='localhost'
    user='Local_Editor'
    password='QuantumGalaxy'
    database='qgdbs'
    mysql=MYSQL(host,user,password,database)
    day=date.today()-timedelta(1)
    day=day.strftime('%Y-%m-%d')
    r=get_query_list(mysql)
    r=query(r,day)
    assert r.errorcode==0
    r=r.data #r->edb_data
    if r.shape==(0,0):
        logger.info('no n_data for day:%s'%day)
        return 'no data today'
    elif r.shape==(8,5):    
        val0=[]
        for i,row in r.iterrows():
            
            val0.append((row['id'],row['time'],float(row['value'])))
        #print(val0)
        d=compute(r) #d-> customized_edb_data

        val1=[]
        for k,v in d.items():
            val1.append((k,v,day))
        #print(val1)
        sql0="insert into edb_data (code,date,value) values(%s,%s,%s)"
        sql1="insert ignore into customized_edb_data (name,value,date) values(%s,%s,%s)"
        result0=mysql.wirte_many_query(sql0,val0)
        result1=mysql.wirte_many_query(sql1,val1)
        r=feishu_work(mysql,day)
        mysql.close()
        #print(result0,result1)
    else:
        logger.info('got error data shape: %s,%s'%r.shape)


In [11]:
job()

NameError: name 'get_query_list' is not defined

In [18]:

from apscheduler.schedulers.background import BackgroundScheduler
job_defaults = { 'max_instances': 10 }
scheduler = BackgroundScheduler(job_defaults=job_defaults)

scheduler.start()
scheduler.add_job(job, 'cron', day_of_week='mon-sun', hour=0, minute=28)

e:\ProgramData\Anaconda3\envs\QGIBG\lib\site-packages\apscheduler\util.py:95: PytzUsageWarning: The zone attribute is specific to pytz's interface; please migrate to a new time zone provider. For more details on how to do so, see https://pytz-deprecation-shim.readthedocs.io/en/latest/migration.html
  if obj.zone == 'local':
e:\ProgramData\Anaconda3\envs\QGIBG\lib\site-packages\apscheduler\triggers\cron\__init__.py:159: PytzUsageWarning: The localize method is no longer necessary, as this time zone supports the fold attribute (PEP 495). For more details on migrating to a PEP 495-compliant implementation, see https://pytz-deprecation-shim.readthedocs.io/en/latest/migration.html
  return self.timezone.localize(datetime(**values))


<Job (id=627b2a82ef3340eeb4b0f036233d3608 name=job)>

In [24]:
scheduler.print_jobs(jobstore=None,out=sys.stdout)

Jobstore default:
    job (trigger: cron[day_of_week='mon-sun', hour='11', minute='0'], next run at: 2022-05-24 11:00:00 CST)


In [12]:
scheduler.shutdown()

NameError: name 'scheduler' is not defined

In [18]:
r=(6,5)
print('%s,%s'%r)

6,5
